# Task4 — Denoise: five-fold validation → audit → frozen Test inference → CSV

**Run All ได้ครบจน export CSV** โดยไม่ฝึกโมเดลใหม่. โหลด folds1–2 ที่ทำเสร็จแล้วจาก cache และทำ folds3–5 เพิ่ม; ตรวจ audit บน Train แล้ว predict Test ด้วย full EfficientNet เดิม

สูตรล็อกไว้: Gaussian radius0.6 ก่อน resize/pad/TTA; compound ResNet25% + sampler ResNet25% + EfficientNetภาพเดิม25% + EfficientNetภาพกรอง25%. ไม่มีการหา sigma/weights จาก Test และไม่ปรับสูตรตาม audit

ใช้ baseline Public0.81047 และคง Task1/2/3/5 เดิมทุกค่า. Export เป็นไฟล์ใหม่ ไม่ทับ baseline ไม่ส่งขึ้น Kaggle อัตโนมัติ

ตามคำขอ **จะ export candidate แม้ validation ไม่ถึงเกณฑ์** พร้อมรายงาน recommend_submission=False; การมีไฟล์ไม่ได้แปลว่าชนะ baseline. เกณฑ์เดิม: original≥−.001, stress family≥+.002 และชนะ≥4/5, worst class≥−.01, worst condition≥−.02; audit mean≥0/worst condition≥−.02

OOF/auditเคยใช้ซ้ำ จึงเป็น exploratory. ไม่รับรอง Public score. ไม่ต้อง full refit เพราะใช้โมเดลเดิมทุกตัว. Cache inference รายcondition/view; หยุดกลางviewจะทำviewนั้นใหม่

ถ้าต้องการหยุดหลังvalidationก่อนเปิดTest ตั้ง EXPORT_SUBMISSION=False ในเซลล์แรก

In [1]:
from pathlib import Path
import sys, json, hashlib, time
import numpy as np
import pandas as pd
import torch
from dataclasses import asdict
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Task4/models/image_training.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from Task4.models import image_diversity_training as training
from Task4.models import image_denoise_evaluation as evaluation
from Task4.models import image_compound_validation as validation
from Task4.models.image_audit import make_folds
from Task4.models.image_compound_transforms import CONDITIONS
RAW=ROOT/'Task4/data/raw'
AUDIT=ROOT/'Task4/data/audit/followup_v2'
OUT=ROOT/'Task4/models/results/denoise_pilot_v1'
BATCH_SIZE=32
NUM_WORKERS=0
print('Device:',training.get_device())
if training.get_device().type=='cpu':raise RuntimeError('CPU training would be slow: select the project kernel with MPS/CUDA available.')

from Task4.tools import denoise_submission as exporter
EXPORT_SUBMISSION=True

Device: mps


## 1. ตรวจ Train, folds และ checkpoint provenance

In [2]:
train=pd.read_csv(RAW/'train.csv')
groups=pd.read_parquet(AUDIT/'train_groups.parquet')
assert train.id.is_unique and train.label.notna().all()
assert all(np.array_equal(train[c],groups[c]) for c in ['id','file_name','label'])
metadata=pd.read_parquet(AUDIT/'images.parquet',filters=[('split','=','train')]).set_index('id').loc[train.id]
assert metadata.split.eq('train').all()
inventory=hashlib.sha256()
for row in metadata.itertuples():
    assert Path(row.file_name).name==row.file_name
    digest=validation.sha(RAW/'train'/row.file_name)
    assert digest==row.sha256, f'Changed Train image: {row.file_name}'
    inventory.update(f'{row.file_name}/{digest}'.encode())
folds=make_folds(groups,mode='near',n_splits=5,seed=42)
fold_hash=hashlib.sha256(b''.join(v.astype('int64').tobytes() for _,v in folds)).hexdigest()
assert fold_hash==json.loads((ROOT/'Task4/models/results/heavy_balance_v1/settings.json').read_text())['fold_hash']
reference_hashes={}
for fold,(fit,valid) in enumerate(folds,1):
    assert not np.intersect1d(fit,valid).size
    assert not set(groups.iloc[fit].near_group)&set(groups.iloc[valid].near_group)
    for path in evaluation.references(ROOT,fold):
        signature=json.loads((path.parent/'signature.json').read_text())
        assert signature['data_signature']==validation.sha_frame(groups.iloc[fit])
        reference_hashes[str(path.relative_to(ROOT))]=validation.sha(path)
        frame=pd.read_parquet(path.parent/'oof.parquet')
        assert frame.id.tolist()==groups.iloc[valid].id.tolist()
        assert np.array_equal(frame.true,groups.iloc[valid].label)
        reference_hashes[str((path.parent/'oof.parquet').relative_to(ROOT))]=validation.sha(path.parent/'oof.parquet')
sources={p.name:validation.sha(p) for p in (ROOT/'Task4/models').glob('image_*.py')}
settings={'recipe':{'filter':'GaussianBlur','radius':0.6,'blend_weights':[.25,.25,.25,.25],'folds':[1,2]},'batch_size':BATCH_SIZE,'workers':NUM_WORKERS,
          'train_manifest':validation.sha(RAW/'train.csv'),'inventory':inventory.hexdigest(),'fold_hash':fold_hash,
          'references':reference_hashes,'sources':sources,'torch':str(torch.__version__),'device':str(training.get_device())}
OUT.mkdir(parents=True,exist_ok=True)
if (OUT/'settings.json').exists():assert json.loads((OUT/'settings.json').read_text())==settings,'Settings changed: use a new OUT'
else:
    assert not any(OUT.iterdir()),'Nonempty OUT without settings'
    validation.write_json(OUT/'settings.json',settings)
print('Train verified:',len(train),'rows; Test unopened')

Train verified: 31546 rows; Test unopened


## 2. Development ครบ 5 folds
Folds1–2 โหลดผล prediction cache; ไม่มีการฝึกซ้ำ

In [3]:
parts=[]
for fold in range(1,6):
    started=time.monotonic()
    score=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold)
    parts.append(score)
    display(evaluation.summarize(score))
    print('Fold',fold,'minutes:',round((time.monotonic()-started)/60,1))
combined=pd.concat(parts,ignore_index=True)
assert set(combined.fold)==set(range(1,6))
assert not combined.duplicated(['fold','condition','procedure']).any()
combined.to_parquet(OUT/'five_fold_scores.parquet',index=False)
display(evaluation.summarize(combined))
s=combined.copy()
s['family']=s.condition.map(lambda c:'original' if c=='original' else ('compound' if c.startswith('compound_') else CONDITIONS[c][1]))
f=s.groupby(['procedure','fold','family']).macro_F1.mean()
delta=(f.loc['denoise_blend25']-f.loc['baseline']).unstack('family')
gains=delta[['original']].assign(stress_family_gain=delta.drop(columns='original').mean(axis=1))
display(gains)
a=combined[combined.procedure.eq('denoise_blend25')].set_index(['fold','condition'])
b=combined[combined.procedure.eq('baseline')].set_index(['fold','condition'])
worst_class=min(float((a.xs('original',level='condition')[f'F1_{cl}']-b.xs('original',level='condition')[f'F1_{cl}']).mean()) for cl in training.CLASS_NAMES)
worst_condition=float((a.macro_F1-b.macro_F1).groupby('condition').mean().min())
development={'original_gain':float(gains.original.mean()),'stress_family_gain':float(gains.stress_family_gain.mean()),
             'wins':int((gains.stress_family_gain>0).sum()),'worst_class_gain':worst_class,'worst_condition_gain':worst_condition}
development['passed']=bool(development['original_gain']>=-.001 and development['stress_family_gain']>=.002 and development['wins']>=4 and worst_class>=-.01 and worst_condition>=-.02)
validation.write_json(OUT/'five_fold_development.json',development)
display(development)

suite,compound,original,single
procedure,,,
baseline,0.944725,0.986331,0.977201
denoise_blend25,0.946052,0.986241,0.977799


Fold 1 minutes: 0.0


suite,compound,original,single
procedure,,,
baseline,0.955820,0.983375,0.9856
denoise_blend25,0.956457,0.983718,0.9858


Fold 2 minutes: 0.0
stress original original
stress stress gaussian_1
stress stress gaussian_2
stress stress speckle_1
stress stress speckle_2
stress stress blur_1
stress stress blur_2
stress stress jpeg_1
stress stress jpeg_2
stress stress dark_1
stress stress dark_2
stress stress compound_dark_noise
stress stress compound_color_noise
stress stress compound_rotation_dark
stress stress compound_bright_color
stress stress compound_dark_color_noise
stress stress compound_rotation_color_noise


suite,compound,original,single
procedure,,,
baseline,0.948277,0.984449,0.981614
denoise_blend25,0.948985,0.983534,0.981815


Fold 3 minutes: 3.9
stress original original
stress stress gaussian_1
stress stress gaussian_2
stress stress speckle_1
stress stress speckle_2
stress stress blur_1
stress stress blur_2
stress stress jpeg_1
stress stress jpeg_2
stress stress dark_1
stress stress dark_2
stress stress compound_dark_noise
stress stress compound_color_noise
stress stress compound_rotation_dark
stress stress compound_bright_color
stress stress compound_dark_color_noise
stress stress compound_rotation_color_noise


... (3 outputs left out)

Fold 5 minutes: 3.8


suite,compound,original,single
procedure,,,
baseline,0.946368,0.983914,0.981367
denoise_blend25,0.946708,0.983909,0.981449


family,original,stress_family_gain
fold,,
1,-0.000090,0.000719
2,0.000343,0.000273
3,-0.000915,0.000285
4,-0.000083,-0.000155
5,0.000721,-0.000498


{'original_gain': -4.924015025564721e-06,
 'stress_family_gain': 0.00012471584533746045,
 'wins': 3,
 'worst_class_gain': -0.0005633276445836221,
 'worst_condition_gain': -0.001199867243974828,
 'passed': False}

## 3. Audit — ไม่มีการเปลี่ยนสูตร
ทำเพื่อรายงานก่อน export แม้ development ไม่ผ่าน ไม่เลือกสูตรจาก audit

In [4]:
audit_scores=exporter.audit(ROOT,RAW,groups,folds,OUT)
a=audit_scores[audit_scores.procedure.eq('denoise_blend25')].set_index(['fold','condition']).macro_F1
b=audit_scores[audit_scores.procedure.eq('baseline')].set_index(['fold','condition']).macro_F1
d=a-b
audit_result={'mean_gain':float(d.mean()),'worst_condition_gain':float(d.groupby('condition').mean().min())}
audit_result['passed']=bool(audit_result['mean_gain']>=0 and audit_result['worst_condition_gain']>=-.02)
review={'folds':[1,2,3,4,5],'development':development,'audit':audit_result,
        'recommend_submission':bool(development['passed'] and audit_result['passed']),
        'scores_sha256':validation.sha(OUT/'five_fold_scores.parquet'),
        'audit_sha256':validation.sha(OUT/'audit_scores.parquet'),
        'test_used_for_selection':False,'formula':'baseline - .25*original_EfficientNet + .25*denoised_EfficientNet'}
validation.write_json(OUT/'five_fold_review.json',review)
display(audit_scores.groupby('procedure').macro_F1.mean())
display(review)
if not review['recommend_submission']:
    print('Validation ยังไม่สนับสนุนให้เปลี่ยน baseline; CSV ที่จะ export เป็น experimental candidate ตามคำขอ')

stress stress shot_1
stress stress shot_2
stress stress impulse_1
stress stress impulse_2
stress stress compound_audit_dark_jpeg
stress stress compound_audit_color_blur


KeyboardInterrupt: 

## 4. Predict Test และ export submission
ใช้ full checkpoints ที่มีอยู่แล้ว ไม่fit/ปรับพารามิเตอร์จากTest. บันทึกcache5views; ตรวจID, label0–4, baselinehash และทุกคอลัมน์อื่นก่อนเขียน CSV

In [ ]:
if EXPORT_SUBMISSION:
    submission_path=exporter.predict_export(ROOT,OUT)
    from IPython.display import FileLink
    display(FileLink(str(submission_path)))
else:
    print('Export disabled: validation completed; Test not opened by this step.')